In [0]:
# Importar bibliotecas necessárias
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Definir URL JDBC
jdbc_url = "jdbc:sqlserver://srvsqlpostech.database.windows.net:1433;databaseName=postechfase3"

# Configurar credenciais
connection_properties = {
    "user": "",
    "password": "",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Ler dados da tabela covid_data_fact
df_covid = spark.read.jdbc(url=jdbc_url, table="covid_data_fact", properties=connection_properties)

# Exibir o esquema original
print("Esquema original:")
df_covid.printSchema()

Esquema original:
root
 |-- Ano: short (nullable = true)
 |-- UF: short (nullable = true)
 |-- CAPITAL: short (nullable = true)
 |-- RM_RIDE: short (nullable = true)
 |-- Numero_selecao_domicilio_V1008: short (nullable = true)
 |-- Semana_no_mes_V1012: short (nullable = true)
 |-- Mes_pesquisa_V1013: short (nullable = true)
 |-- Num_entrevista_domicilio_V1016: short (nullable = true)
 |-- Estrato: string (nullable = true)
 |-- UPA: long (nullable = true)
 |-- Situacao_domicilio_V1022: short (nullable = true)
 |-- Tipo_Area_V1023: short (nullable = true)
 |-- Projecao_Populacao_V1030: integer (nullable = true)
 |-- Peso_Domicilio_sem_estrat_V1031: decimal(18,2) (nullable = true)
 |-- Peso_Domicilio_com_estrat_V1032: decimal(18,2) (nullable = true)
 |-- posest: short (nullable = true)
 |-- nro_ordem_A001: short (nullable = true)
 |-- Condicao_no_domicilio_A001A: short (nullable = true)
 |-- Dia_nascimento_A001B1: short (nullable = true)
 |-- Mes_nascimento_A001B2: short (nullable = true)

In [0]:
# Função para mapear códigos para valores descritivos
def map_values(df, column, mapping_dict):
    return df.withColumn(column, F.when(F.col(column).isNull(), None).otherwise(F.create_map([F.lit(x) for x in sum(mapping_dict.items(), ())])[F.col(column)]))

# 1. Renomear colunas para nomes mais descritivos
df_renamed = df_covid \
    .withColumnRenamed("Ano", "ANO") \
    .withColumnRenamed("UF", "CODIGO_UF") \
    .withColumnRenamed("CAPITAL", "CODIGO_CAPITAL") \
    .withColumnRenamed("RM_RIDE", "CODIGO_REGIAO_METROPOLITANA") \
    .withColumnRenamed("Semana_no_mes_V1012", "SEMANA_MES") \
    .withColumnRenamed("Mes_pesquisa_V1013", "MES_PESQUISA") \
    .withColumnRenamed("Situacao_domicilio_V1022", "CODIGO_SITUACAO_DOMICILIO") \
    .withColumnRenamed("Tipo_Area_V1023", "CODIGO_TIPO_AREA") \
    .withColumnRenamed("Idade_A002", "IDADE") \
    .withColumnRenamed("Sexo_A003", "CODIGO_SEXO") \
    .withColumnRenamed("Cor_raca_A004", "CODIGO_COR_RACA") \
    .withColumnRenamed("Escolaridade_A005", "CODIGO_ESCOLARIDADE") \
    .withColumnRenamed("Febre_B0011", "CODIGO_TEVE_FEBRE") \
    .withColumnRenamed("Tosse_B0012", "CODIGO_TEVE_TOSSE") \
    .withColumnRenamed("Dor_Garganta_B0013", "CODIGO_TEVE_DOR_GARGANTA") \
    .withColumnRenamed("Dificuldade_respiracao_B0014", "CODIGO_TEVE_FALTA_AR") \
    .withColumnRenamed("Dor_cabeca_B0015", "CODIGO_TEVE_DOR_CABECA") \
    .withColumnRenamed("Dor_Peito_B0016", "CODIGO_TEVE_DOR_PEITO") \
    .withColumnRenamed("Nausea_B0017", "CODIGO_TEVE_NAUSEA") \
    .withColumnRenamed("Nariz_Entupido_escorrendo_B0018", "CODIGO_TEVE_NARIZ_ENTUPIDO") \
    .withColumnRenamed("Fadiga_B0019", "CODIGO_TEVE_FADIGA") \
    .withColumnRenamed("Dor_olhos_B00110", "CODIGO_TEVE_DOR_OLHOS") \
    .withColumnRenamed("Perda_cheiro_B00111", "CODIGO_TEVE_PERDA_PALADAR") \
    .withColumnRenamed("Dor_muscular_B00112", "CODIGO_TEVE_DOR_MUSCULAR") \
    .withColumnRenamed("Foi_estabelecimento_saude_B002", "CODIGO_FOI_AO_MEDICO") \
    .withColumnRenamed("sedado_entubado_quando_internado_B006", "CODIGO_FOI_SEDADO_ENTUBADO") \
    .withColumnRenamed("plano_saude_B007", "CODIGO_TEM_PLANO_SAUDE") \
    .withColumnRenamed("trabalho_remoto_C013", "CODIGO_TRABALHO_REMOTO") \
    .withColumnRenamed("auxilio_emergencial_covid_D0051", "CODIGO_AUXILIO_EMERGENCIAL_COVID")

In [0]:
# 2. Criar mapeamentos para códigos
# Mapeamento UF
uf_mapping = {
    11: "Rondônia", 12: "Acre", 13: "Amazonas", 14: "Roraima", 15: "Pará", 16: "Amapá", 17: "Tocantins",
    21: "Maranhão", 22: "Piauí", 23: "Ceará", 24: "Rio Grande do Norte", 25: "Paraíba", 26: "Pernambuco",
    27: "Alagoas", 28: "Sergipe", 29: "Bahia", 31: "Minas Gerais", 32: "Espírito Santo", 33: "Rio de Janeiro",
    35: "São Paulo", 41: "Paraná", 42: "Santa Catarina", 43: "Rio Grande do Sul", 50: "Mato Grosso do Sul",
    51: "Mato Grosso", 52: "Goiás", 53: "Distrito Federal"
}

# Mapeamento Situação Domicílio
situacao_domicilio_mapping = {1: "Urbana", 2: "Rural"}

# Mapeamento Tipo Área
tipo_area_mapping = {
    1: "Capital",
    2: "Resto da RM (Região Metropolitana, excluindo a capital)",
    3: "Resto da RIDE (Região Integrada de Desenvolvimento Econômico, excluindo a capital)",
    4: "Resto da UF (Unidade da Federação, excluindo a região metropolitana e a RIDE)"
}

# Mapeamento Sexo
sexo_mapping = {1: "Homem", 2: "Mulher"}

# Mapeamento Cor/Raça
cor_raca_mapping = {1: "Branca", 2: "Preta", 3: "Amarela", 4: "Parda", 5: "Indígena", 9: "Ignorado"}

# Mapeamento Escolaridade
escolaridade_mapping = {
    1: "Sem instrução",
    2: "Fundamental incompleto",
    3: "Fundamental completa",
    4: "Médio incompleto",
    5: "Médio completo",
    6: "Superior incompleto",
    7: "Superior completo",
    8: "Pós-graduação, mestrado ou doutorado"
}

# Mapeamento Sim/Não/Ignorado para sintomas e outras perguntas
sim_nao_mapping = {1: "Sim", 2: "Não", 3: "Não sabe", 9: "Ignorado"}

# Mapeamento para trabalho remoto
trabalho_remoto_mapping = {1: "Sim", 2: "Não", 9: "Não aplicável"}

# Mapeamento para auxílio emergencial
auxilio_emergencial_mapping = {1: "Sim", 2: "Não"}

In [0]:
# 3. Aplicar mapeamentos para criar colunas descritivas
df_mapped = df_renamed
df_mapped = map_values(df_mapped, "CODIGO_UF", uf_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_UF", "UF")

df_mapped = map_values(df_mapped, "CODIGO_SITUACAO_DOMICILIO", situacao_domicilio_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_SITUACAO_DOMICILIO", "SITU_DOMICILIO")

df_mapped = map_values(df_mapped, "CODIGO_TIPO_AREA", tipo_area_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_TIPO_AREA", "TIPO_AREA")

df_mapped = map_values(df_mapped, "CODIGO_SEXO", sexo_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_SEXO", "SEXO")

df_mapped = map_values(df_mapped, "CODIGO_COR_RACA", cor_raca_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_COR_RACA", "COR_RACA")

df_mapped = map_values(df_mapped, "CODIGO_ESCOLARIDADE", escolaridade_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_ESCOLARIDADE", "ESCOLARIDADE")

# Aplicar mapeamento para sintomas
sintomas_cols = [
    "CODIGO_TEVE_FEBRE", "CODIGO_TEVE_TOSSE", "CODIGO_TEVE_DOR_GARGANTA",
    "CODIGO_TEVE_FALTA_AR", "CODIGO_TEVE_DOR_CABECA", "CODIGO_TEVE_DOR_PEITO",
    "CODIGO_TEVE_NAUSEA", "CODIGO_TEVE_NARIZ_ENTUPIDO", "CODIGO_TEVE_FADIGA",
    "CODIGO_TEVE_DOR_OLHOS", "CODIGO_TEVE_PERDA_PALADAR", "CODIGO_TEVE_DOR_MUSCULAR"
]

for col in sintomas_cols:
    new_col = col.replace("CODIGO_TEVE_", "SE_TEVE_")
    df_mapped = map_values(df_mapped, col, sim_nao_mapping)
    df_mapped = df_mapped.withColumnRenamed(col, new_col)

# Aplicar mapeamento para outras colunas
df_mapped = map_values(df_mapped, "CODIGO_FOI_AO_MEDICO", sim_nao_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_FOI_AO_MEDICO", "FOI_AO_MEDICO")

df_mapped = map_values(df_mapped, "CODIGO_FOI_SEDADO_ENTUBADO", sim_nao_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_FOI_SEDADO_ENTUBADO", "FOI_SEDADO_ENTUBADO")

df_mapped = map_values(df_mapped, "CODIGO_TEM_PLANO_SAUDE", sim_nao_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_TEM_PLANO_SAUDE", "TEM_PLANO_SAUDE")

df_mapped = map_values(df_mapped, "CODIGO_TRABALHO_REMOTO", trabalho_remoto_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_TRABALHO_REMOTO", "TRABALHO_REMOTO")

df_mapped = map_values(df_mapped, "CODIGO_AUXILIO_EMERGENCIAL_COVID", auxilio_emergencial_mapping)
df_mapped = df_mapped.withColumnRenamed("CODIGO_AUXILIO_EMERGENCIAL_COVID", "AUXILIO_EMERGENCIAL_COVID")

In [0]:
# 4. Criar coluna ANOMES
df_mapped = df_mapped.withColumn("ANOMES", F.concat(F.col("ANO"), F.lpad(F.col("MES_PESQUISA"), 2, "0")))

In [0]:
# 5. Criar coluna REGIAO baseada na UF
df_mapped = df_mapped.withColumn("REGIAO",
    F.when(F.col("UF").isin(["Rondônia", "Acre", "Amazonas", "Roraima", "Pará", "Amapá", "Tocantins"]), "Norte")
     .when(F.col("UF").isin(["Maranhão", "Piauí", "Ceará", "Rio Grande do Norte", "Paraíba", "Pernambuco", "Alagoas", "Sergipe", "Bahia"]), "Nordeste")
     .when(F.col("UF").isin(["Minas Gerais", "Espírito Santo", "Rio de Janeiro", "São Paulo"]), "Sudeste")
     .when(F.col("UF").isin(["Paraná", "Santa Catarina", "Rio Grande do Sul"]), "Sul")
     .when(F.col("UF").isin(["Mato Grosso do Sul", "Mato Grosso", "Goiás", "Distrito Federal"]), "Centro-Oeste")
     .otherwise(None))

In [0]:
# 6. Criar coluna FAIXA_ETARIA
df_mapped = df_mapped.withColumn("FAIXA_ETARIA",
    F.when(F.col("IDADE") < 12, "Criança")
     .when((F.col("IDADE") >= 12) & (F.col("IDADE") < 18), "Adolescente")
     .when((F.col("IDADE") >= 18) & (F.col("IDADE") < 25), "Jovem")
     .when((F.col("IDADE") >= 25) & (F.col("IDADE") < 60), "Adulto")
     .when(F.col("IDADE") >= 60, "Idoso")
     .otherwise(None))

In [0]:
# 7. Criar features adicionais
# Indicador de sintomas graves
df_mapped = df_mapped.withColumn("SINTOMAS_GRAVES",
    F.when((F.col("SE_TEVE_FALTA_AR") == "Sim") &
           (F.col("SE_TEVE_FEBRE") == "Sim") &
           (F.col("SE_TEVE_TOSSE") == "Sim"), "Sim")
     .otherwise("Não"))

# Indicador de vulnerabilidade (idosos sem plano de saúde)
df_mapped = df_mapped.withColumn("VULNERAVEL",
    F.when((F.col("FAIXA_ETARIA") == "Idoso") &
           (F.col("TEM_PLANO_SAUDE") == "Não"), "Sim")
     .otherwise("Não"))

# Indicador de nível de risco
df_mapped = df_mapped.withColumn("NIVEL_RISCO",
    F.when((F.col("FAIXA_ETARIA") == "Idoso") &
           (F.col("SINTOMAS_GRAVES") == "Sim"), "Muito Alto")
     .when((F.col("FAIXA_ETARIA") == "Idoso") &
           (F.col("TEM_PLANO_SAUDE") == "Não"), "Alto")
     .when((F.col("FAIXA_ETARIA") == "Idoso") |
           (F.col("SINTOMAS_GRAVES") == "Sim"), "Médio")
     .otherwise("Baixo"))

In [0]:
# 8. Selecionar colunas relevantes para o DataFrame final
columns_to_keep = [
    "ANOMES", "ANO", "MES_PESQUISA", "SEMANA_MES", "UF", "REGIAO",
    "SITU_DOMICILIO", "TIPO_AREA", "IDADE", "FAIXA_ETARIA", "SEXO",
    "COR_RACA", "ESCOLARIDADE",
    "SE_TEVE_FEBRE", "SE_TEVE_TOSSE", "SE_TEVE_DOR_GARGANTA",
    "SE_TEVE_FALTA_AR", "SE_TEVE_DOR_CABECA", "SE_TEVE_DOR_PEITO",
    "SE_TEVE_NAUSEA", "SE_TEVE_NARIZ_ENTUPIDO", "SE_TEVE_FADIGA",
    "SE_TEVE_DOR_OLHOS", "SE_TEVE_PERDA_PALADAR", "SE_TEVE_DOR_MUSCULAR",
    "FOI_AO_MEDICO", "FOI_SEDADO_ENTUBADO", "TEM_PLANO_SAUDE",
    "TRABALHO_REMOTO", "AUXILIO_EMERGENCIAL_COVID",
    "SINTOMAS_GRAVES", "VULNERAVEL", "NIVEL_RISCO"
]

df_final = df_mapped.select([col for col in columns_to_keep if col in df_mapped.columns])

# Exibir o esquema final
print("\nEsquema após tratamento:")
df_final.printSchema()

# Exibir amostra dos dados tratados
print("\nAmostra dos dados tratados:")
display(df_final.limit(5))

# Análises básicas
print("\nContagem de registros por região:")
display(df_final.groupBy("REGIAO").count().orderBy("count", ascending=False))

print("\nDistribuição por faixa etária:")
display(df_final.groupBy("FAIXA_ETARIA").count().orderBy("count", ascending=False))

print("\nDistribuição por nível de risco:")
display(df_final.groupBy("NIVEL_RISCO").count().orderBy("count", ascending=False))

print("\nSintomas mais comuns:")
sintomas_cols = [col for col in df_final.columns if col.startswith("SE_TEVE_")]
for col in sintomas_cols:
    display(df_final.filter(F.col(col) == "Sim").groupBy(col).count())

# Salvar o DataFrame tratado
df_final.write.format("delta").mode("overwrite").saveAsTable("covid_data_tratado")
print("\nDataFrame tratado salvo como tabela Delta 'covid_data_tratado'")


Esquema após tratamento:
root
 |-- ANOMES: string (nullable = true)
 |-- ANO: short (nullable = true)
 |-- MES_PESQUISA: short (nullable = true)
 |-- SEMANA_MES: short (nullable = true)
 |-- UF: string (nullable = true)
 |-- REGIAO: string (nullable = true)
 |-- SITU_DOMICILIO: string (nullable = true)
 |-- TIPO_AREA: string (nullable = true)
 |-- IDADE: short (nullable = true)
 |-- FAIXA_ETARIA: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- COR_RACA: string (nullable = true)
 |-- ESCOLARIDADE: string (nullable = true)
 |-- SE_TEVE_FEBRE: string (nullable = true)
 |-- SE_TEVE_TOSSE: string (nullable = true)
 |-- SE_TEVE_DOR_GARGANTA: string (nullable = true)
 |-- SE_TEVE_FALTA_AR: string (nullable = true)
 |-- SE_TEVE_DOR_CABECA: string (nullable = true)
 |-- SE_TEVE_DOR_PEITO: string (nullable = true)
 |-- SE_TEVE_NAUSEA: string (nullable = true)
 |-- SE_TEVE_NARIZ_ENTUPIDO: string (nullable = true)
 |-- SE_TEVE_FADIGA: string (nullable = true)
 |-- SE_TEVE_DOR_OL

ANOMES,ANO,MES_PESQUISA,SEMANA_MES,UF,REGIAO,SITU_DOMICILIO,TIPO_AREA,IDADE,FAIXA_ETARIA,SEXO,COR_RACA,ESCOLARIDADE,SE_TEVE_FEBRE,SE_TEVE_TOSSE,SE_TEVE_DOR_GARGANTA,SE_TEVE_FALTA_AR,SE_TEVE_DOR_CABECA,SE_TEVE_DOR_PEITO,SE_TEVE_NAUSEA,SE_TEVE_NARIZ_ENTUPIDO,SE_TEVE_FADIGA,SE_TEVE_DOR_OLHOS,SE_TEVE_PERDA_PALADAR,SE_TEVE_DOR_MUSCULAR,FOI_AO_MEDICO,FOI_SEDADO_ENTUBADO,TEM_PLANO_SAUDE,TRABALHO_REMOTO,AUXILIO_EMERGENCIAL_COVID,SINTOMAS_GRAVES,VULNERAVEL,NIVEL_RISCO
202005,2020,5,3,Maranhão,Nordeste,Rural,"Resto da UF (Unidade da Federação, excluindo a região metropolitana e a RIDE)",33,Adulto,Mulher,Branca,Fundamental incompleto,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,null,null,Não,null,Sim,Não,Não,Baixo
202005,2020,5,3,Maranhão,Nordeste,Rural,"Resto da UF (Unidade da Federação, excluindo a região metropolitana e a RIDE)",9,Criança,Homem,Parda,Fundamental incompleto,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,null,null,Não,null,Sim,Não,Não,Baixo
202005,2020,5,3,Maranhão,Nordeste,Rural,"Resto da UF (Unidade da Federação, excluindo a região metropolitana e a RIDE)",4,Criança,Homem,Branca,Sem instrução,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,null,null,Não,null,Sim,Não,Não,Baixo
202005,2020,5,3,Maranhão,Nordeste,Rural,"Resto da UF (Unidade da Federação, excluindo a região metropolitana e a RIDE)",39,Adulto,Homem,Parda,Sem instrução,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,null,null,Não,null,Sim,Não,Não,Baixo
202005,2020,5,3,Maranhão,Nordeste,Rural,"Resto da UF (Unidade da Federação, excluindo a região metropolitana e a RIDE)",32,Adulto,Mulher,Parda,Fundamental incompleto,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,null,null,Não,null,Sim,Não,Não,Baixo



Contagem de registros por região:


REGIAO,count
Nordeste,336264
Sudeste,332210
Sul,192162
Norte,135435
Centro-Oeste,118671



Distribuição por faixa etária:


...

SE_TEVE_FADIGA,count
Sim,15067


SE_TEVE_DOR_OLHOS,count
Sim,11206


SE_TEVE_PERDA_PALADAR,count
Sim,13946


SE_TEVE_DOR_MUSCULAR,count
Sim,25318



DataFrame tratado salvo como tabela Delta 'covid_data_tratado'
